# DeepReviewer Tutorial: Automated Peer Review with Large Language Models

Welcome to the DeepReviewer tutorial! This notebook will guide you through using DeepReviewer, a powerful tool for generating automated peer reviews of academic papers. DeepReviewer leverages large language models (LLMs) to provide structured, comprehensive feedback, simulating the insights of multiple human reviewers.

**Key Features:**

*   **Structured Reviews:** Generates reviews with sections like Summary, Soundness, Presentation, Contribution, Strengths, Weaknesses, Suggestions, Questions, Rating, and Confidence.
*   **Multiple Review Modes:** Offers "Fast", "Standard", and "Best" modes to balance speed and detail.
*   **Simulated Reviewers:** "Standard" and "Best" modes simulate multiple reviewers for diverse perspectives.
*   **Meta-Review Generation:** Combines individual reviewer feedback into a concise meta-review.
*   **Structured Output:** Provides results in an easily parsable format.
* **Customizable Model:** Allows you to use different pre-trained DeepReviewer models or your fine-tuned models.

**This tutorial will cover:**

1.  Setting up the environment and installing dependencies.
2.  Loading a paper for review.
3.  Generating reviews in different modes (Fast, Standard).
4.  Parsing the review results to extract structured feedback.

## 1. Setup and Installation

First, we need to install the required libraries.  DeepReviewer relies on `transformers` and `vllm` for efficient model loading and inference. Run the following commands in your terminal or in a notebook cell:

```bash
pip install transformers
pip install vllm
```

Now, let's import the `DeepReviewer` class and initialize the model.

In [1]:
from ai_researcher.deep_reviewer import DeepReviewer

# Initialize DeepReviewer with optimized parameters for 14B model
# Reduced max_model_len and gpu_memory_utilization to fit in GPU memory

reviewer = DeepReviewer(
    custom_model_name="WestlakeNLP/DeepReviewer-7B", 
    device="cuda", 
    tensor_parallel_size=1, 
    gpu_memory_utilization=0.85  # Reduced from 0.95 to leave more headroom
)

# Other parameters you can customize:
# - custom_model_name:  Path to a custom DeepReviewer model (overrides model_size).
# - tensor_parallel_size:  Number of GPUs to use for parallel processing (for larger models).
# - gpu_memory_utilization:  Fraction of GPU memory to use (0.85 is safer than 0.95).
# - max_model_len: Maximum sequence length the model can handle (32768 is sufficient for most papers).

INFO 01-27 15:43:24 [utils.py:263] non-default args: {'max_model_len': 70000, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'WestlakeNLP/DeepReviewer-7B'}
INFO 01-27 15:43:26 [model.py:530] Resolved architecture: Qwen2ForCausalLM
INFO 01-27 15:43:26 [model.py:1545] Using max model len 70000
INFO 01-27 15:43:27 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 01-27 15:43:27 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-27 15:43:27 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
(EngineCore_DP0 pid=2086480) INFO 01-27 15:43:29 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='WestlakeNLP/DeepReviewer-7B', speculative_config=None, tokenizer='WestlakeNLP/DeepReviewer-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=70000, download_dir=None, load_format=auto, tensor_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2086480) INFO 01-27 15:43:56 [default_loader.py:291] Loading weights took 21.24 seconds
(EngineCore_DP0 pid=2086480) INFO 01-27 15:43:57 [gpu_model_runner.py:3905] Model loading took 14.27 GiB memory and 23.723843 seconds
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:01 [backends.py:644] Using cache directory: /home/zhihang/.cache/vllm/torch_compile_cache/926ee3de98/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:01 [backends.py:704] Dynamo bytecode transform time: 3.80 s
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:03 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.823 s
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:03 [monitor.py:34] torch.compile takes 4.62 s in total
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:04 [gpu_worker.py:358] Available KV cache memory: 10.86 GiB
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:05 [kv_cache_utils.py:1305] GPU KV cache size: 20

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 27.64it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 30.77it/s]


(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:08 [gpu_model_runner.py:4856] Graph capturing finished in 4 secs, took 0.36 GiB
(EngineCore_DP0 pid=2086480) INFO 01-27 15:44:08 [core.py:273] init engine (profile, create kv cache, warmup model) took 11.42 seconds
INFO 01-27 15:44:09 [llm.py:347] Supported tasks: ['generate']


## 2. Loading a Paper for Review

We'll load a paper from a JSON file.  The JSON file should contain a list of papers, where each paper is a dictionary with at least a `title` and `latex` key.  We've provided a sample file named `generated_paper.json` for this tutorial.

In [2]:
import json

# Load the paper(s) from the JSON file
with open('generated_paper.json', 'r', encoding='utf-8') as f:
    papers = json.load(f)

# Print some basic information about the loaded papers
for i, paper in enumerate(papers):
    print(f"Paper {i+1}: Title: {paper['title']}")
    print(f"Paper {i+1}: LaTeX content length: {len(paper['latex'])} characters")
    print('\n')

Paper 1: Title: Improving AI Scientists through Multi-Agent Competitive Preference Optimization


Paper 1: LaTeX content length: 45723 characters


Paper 2: Title: ReviewerNet: Harnessing Multi-Agent Systems for Comprehensive Scientific Paper Review


Paper 2: LaTeX content length: 41659 characters


Paper 3: Title: Reviewer Agent: Enhancing Scientific Peer Review Experience with Reviewer Agents


Paper 3: LaTeX content length: 36850 characters


Paper 4: Title: Scientific Review Agents: A Step Towards Artificial Intelligence-Assisted Peers


Paper 4: LaTeX content length: 43846 characters


Paper 5: Title: Towards Autonomous Scientific Discovery via Multi-Agent Iterative Preference Optimization


Paper 5: LaTeX content length: 45265 characters


Paper 6: Title: Evaluating Language Models on Scientific Writing: Benchmarks, Algorithms, and Case Studies


Paper 6: LaTeX content length: 43432 characters


Paper 7: Title: WAIWIG: Towards OpenAI-assisted Scientific Writing and Inter-Results

## 3. Generating Reviews

Now, let's use DeepReviewer to generate reviews. We'll demonstrate both "Fast Mode" and "Standard Mode."

### 3.1 Fast Mode

In [3]:
# Generate reviews in Fast Mode
fast_review_results = reviewer.evaluate([paper['latex'] for paper in papers], mode="Fast Mode")

# Process and print the results for each paper
for i, review_result in enumerate(fast_review_results):
    print(f"\n--- Fast Mode Review for Paper: {papers[i]['title']} ---")
    if review_result:
        print(f"Raw text: {review_result.get('raw_text', 'N/A')}")  # Show the raw output
    else:
        print("Review generation failed.")

Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Of course. I will use Fast Mode for quick thinking. As a professional reviewer, I will directly output a detailed evaluation of this paper. Let me think - Fast Mode means I will directly output a Summary, followed by scores for Soundness, Presentation and Contribution, then provide analysis of Strengths, Weaknesses, Suggestions, and Questions. Finally, I will output the Rating, Confidence and Decision:

\boxed_review{
## Summary:

This paper introduces Multi-Agent Competitive Preference Optimization (MAPO), a novel framework designed to enhance the quality of AI-driven scientific reviews. The core idea revolves around employing a multi-agent system where multiple AI agents, each with distinct roles, collaboratively evaluate research papers. Specifically, the framework utilizes two types of agents: an 'evaluation agent' that generates initial comments on a paper, and a 'screening agent' that makes a binary decision on whether the paper should be accepted or rejected. These agents are tr

### 3.2 Standard Mode

In [4]:
# Generate reviews in Standard Mode with 3 simulated reviewers
standard_review_results = reviewer.evaluate([paper['latex'] for paper in papers], mode="Standard Mode", reviewer_num=3)

# Process and print the results for each paper
for i, review_result in enumerate(standard_review_results):
    print(f"\n--- Standard Mode Review for Paper: {papers[i]['title']} ---")
    if review_result:
      print(f"Raw text: {review_result.get('raw_text', 'N/A')}") # Show the raw output
    else:
      print('Review generation failed')

Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

I will use Standard Mode for comprehensive thinking. As a professional reviewer, I will simulate 3 different reviewers, followed by the verification thinking. Then I will output the Finally Review Output. Let me think - Standard Mode means I will output the original review, followed by the verification thinking. Considering that I am currently in standard mode, I should think from my existing knowledge and consider some related work content when writing about weaknesses. Then I will output the Finally Review Output and Meta Review Output:

\boxed_simreviewers{
## Reviewer 1

### Summary

This paper proposes a competitive preference optimization approach to improve AI scientists' performance in scientific paper reviews. The authors introduce two agents: a screening agent and an evaluation agent. The screening agent determines whether a paper should be accepted or rejected, while the evaluation agent provides detailed comments or justifications for the screening agent's decision. The aut

## 4. Parsing Review Results
Let's parse and examine the structured information from the Standard Mode reviews. We'll extract key data like the average rating, decision, and the number of reviewers.

In [5]:
# Process and print the parsed results for each paper
for i, review_result in enumerate(standard_review_results):
    print(f"\n************\n------\n************\n--- Parsed Review for Paper: {papers[i]['title']}")
    if review_result:
        # --- Meta-Review (if available) ---
        if review_result['meta_review']:
            print("\n--- Meta-Review ---")
            print(f"  Summary: {review_result['meta_review'].get('summary', 'N/A')}")
            print(f"  Rating: {review_result['meta_review'].get('rating', 'N/A')}")
            print(f"  Soundness: {review_result['meta_review'].get('soundness', 'N/A')}")
            print(f"  Presentation: {review_result['meta_review'].get('presentation', 'N/A')}")
            print(f"  Contribution: {review_result['meta_review'].get('contribution', 'N/A')}")

        # --- Overall Decision (if available) ---
        if review_result['decision']:
            print("\n--- Overall Decision ---")
            print(f"  Decision: {review_result['decision']}")

    else:
        print("Review generation failed.")


************
------
************
--- Parsed Review for Paper: Improving AI Scientists through Multi-Agent Competitive Preference Optimization



--- Meta-Review ---
  Summary: This paper introduces a novel multi-agent competitive preference optimization approach designed to enhance the quality of AI-generated reviews in scientific paper peer review. The core idea revolves around employing two distinct agents: a 'screening agent' that determines whether a paper should be accepted or rejected, and an 'evaluation agent' that provides a detailed comment or justification for the screening agent's decision. The method leverages a competitive preference optimization framework, which aims to maximize the accuracy of paper acceptance decisions by simulating a competitive environment between these two agents. The authors utilize a multi-agent reinforcement learning (MARL) technique, specifically a modified version of the MADDPG algorithm, to train these agents. The training process involves two

Let's parse and examine the structured information from the reviews to identify the best paper. We'll extract key data like the average rating, decision, and individual reviewer scores to determine which paper received the highest evaluation.

In [6]:
# Function to extract average rating from a review
def extract_average_rating(review_result):
    if not review_result:
        return 0.0
    
    # Method 1: Extract from meta_review if available
    if review_result.get('meta_review') and 'rating' in review_result['meta_review']:
        try:
            return float(review_result['meta_review']['rating'])
        except (ValueError, TypeError):
            pass
    
    # Method 2: Calculate average from individual reviewer ratings
    ratings = []
    for review in review_result.get('reviews', []):
        if 'rating' in review:
            try:
                ratings.append(float(review['rating']))
            except (ValueError, TypeError):
                pass
    
    return np.mean(ratings) if ratings else 0.0

# Find the best paper based on average rating
paper_ratings = []
for i, (paper, review) in enumerate(zip(papers, standard_review_results)):
    avg_rating = extract_average_rating(review)
    decision = review.get('decision', 'No decision')
    reviewer_count = len(review.get('reviews', []))
    
    paper_ratings.append({
        'index': i,
        'title': paper['title'],
        'avg_rating': avg_rating,
        'decision': decision,
        'reviewer_count': reviewer_count
    })
    
    print(f"Paper {i+1}: {paper['title']}")
    print(f"  Average Rating: {avg_rating:.2f}/10")
    print(f"  Decision: {decision}")
    print(f"  Number of Reviewers: {reviewer_count}")
    print()

# Identify the best paper
if paper_ratings:
    best_paper = max(paper_ratings, key=lambda x: x['avg_rating'])
    print("=" * 50)
    print(f"BEST PAPER: {best_paper['title']}")
    print(f"Rating: {best_paper['avg_rating']:.2f}/10")
    print(f"Decision: {best_paper['decision']}")
    print("=" * 50)
else:
    print("No valid paper ratings found.")

Paper 1: Improving AI Scientists through Multi-Agent Competitive Preference Optimization


  Average Rating: 5.33/10
  Decision: Reject
  Number of Reviewers: 3

Paper 2: ReviewerNet: Harnessing Multi-Agent Systems for Comprehensive Scientific Paper Review


  Average Rating: 5.33/10
  Decision: Reject
  Number of Reviewers: 3

Paper 3: Reviewer Agent: Enhancing Scientific Peer Review Experience with Reviewer Agents


  Average Rating: 3.67/10
  Decision: Reject
  Number of Reviewers: 3

Paper 4: Scientific Review Agents: A Step Towards Artificial Intelligence-Assisted Peers


  Average Rating: 5.00/10
  Decision: Reject
  Number of Reviewers: 3

Paper 5: Towards Autonomous Scientific Discovery via Multi-Agent Iterative Preference Optimization


  Average Rating: 5.67/10
  Decision: Reject
  Number of Reviewers: 3

Paper 6: Evaluating Language Models on Scientific Writing: Benchmarks, Algorithms, and Case Studies


  Average Rating: 4.67/10
  Decision: Reject
  Number of Reviewers: 3



## 6. Conclusion and Further Exploration

In this tutorial, you've learned how to use DeepReviewer to generate automated peer reviews, explore different review modes, and parse the structured output. DeepReviewer offers a powerful and flexible way to leverage the capabilities of large language models for scientific paper evaluation.

**Limitations and Ethical Considerations:**

*   **Not a Replacement for Human Review:** DeepReviewer is a tool to *assist* with peer review, not to replace the critical thinking and expertise of human reviewers.
*   **Potential for Bias:**  Like all LLMs, DeepReviewer can inherit biases from its training data.  Be mindful of this when interpreting the results.
*   **Over-Reliance:**  Avoid over-reliance on automated feedback.  Always critically evaluate the generated reviews.
* **Transparency:** If using DeepReviewer in your work, disclose its use transparently.

**Further Exploration:**

*   **Experiment with different papers:**  Try DeepReviewer with your own research papers or papers from various fields.
*   **Adjust parameters:**  Explore the effects of `reviewer_num` and `max_tokens`.
*   **Explore the vLLM documentation:**  Learn more about the underlying inference engine for advanced usage.
*   **Contribute to the project:**  If you find issues or have suggestions, consider contributing to the DeepReviewer project (if it's open-source).

Thank you for exploring DeepReviewer! We hope this tutorial has been helpful.